# Creating general practice splits 

These must align with those used in the full pre-training model to ensure any downstream comparison does not lead to data leakage

In [ ]:
import os
from pathlib import Path
import sys
node_type = os.getenv('BB_CPU')
venv_dir = f'/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-{node_type}'
venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    sys.path.insert(0, str(venv_site_pkgs))
    print(f"Added path '{venv_site_pkgs}' at start of search paths.")
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")

!pwd

%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from hydra import compose, initialize
from omegaconf import OmegaConf
import logging
import time
import pickle 

from FastEHR.dataloader import FoundationalDataModule
from FastEHR.database.collector import SQLiteDataCollector

from SurvivEHR.examples.data.study_criteria import t2d_inclusion_method

torch.manual_seed(1337)

logging.basicConfig(level=logging.INFO)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = "cpu"    # if more informative debugging statements are needed
print(f"Using device: {device}.")


## Initialise a collector to interact with the database

We need this as we want to filter our practice inclusion to create regional splits

In [ ]:
PATH_TO_DB = "/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/cprd.db"

collector = SQLiteDataCollector(db_path=PATH_TO_DB)


def reduce_by_health_authority(practice_ids, 
                               health_auth,
                               collector):
    collector.connect()

    
    # reduced_list = []
    # for pid in list_of_practice_ids:
    #     cursor.execute(f"""SELECT HEALTH_AUTH FROM static_table WHERE PRACTICE_ID=='{train_practice_ids[0]}' LIMIT 1""")   # 

    # Use parameterized query with tuple expansion
    placeholders_ids = ",".join("?" for _ in practice_ids)
    placeholder_authorities = ",".join("?" for _ in health_auth)
    query = f"""
        SELECT DISTINCT PRACTICE_ID
        FROM static_table
        WHERE PRACTICE_ID IN ({placeholders_ids})
          AND HEALTH_AUTH in ({placeholder_authorities})
    """
    collector.cursor.execute(query, (*practice_ids, *health_auth))
    filtered_ids = [row[0] for row in collector.cursor.fetchall()]

    collector.disconnect()
    return filtered_ids

### Available health authorities

In [ ]:
collector.connect()
collector.cursor.execute("""PRAGMA table_info(static_table);""")
columns_info = collector.cursor.fetchall()
print([c[1] for c in columns_info])
collector.disconnect()

In [ ]:
collector.connect()

query = f"SELECT DISTINCT HEALTH_AUTH FROM static_table;"
collector.cursor.execute(query)

unique_values = [row[0] for row in collector.cursor.fetchall()]

print(f"Unique health authories:")
for val in unique_values:
    print(val)

collector.disconnect()

### We have already specified training-test-validation splits for general practices in the UK

We must ensure we re-use the same splits to avoid data-leakage

In [ ]:
# Original pre-training splits
overwrite_practice_ids = "/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/PreTrain/practice_id_splits.pickle"
with open(overwrite_practice_ids, 'rb') as f:
    splits = pickle.load(f)

train_practice_ids = splits["train"]
val_practice_ids = splits["val"]
test_practice_ids = splits["test"]
print(f"Initial pre-training splits have {len(train_practice_ids)}, "
      f"{len(val_practice_ids)} and {len(test_practice_ids)} "
      f"training, validation and test practices, respectively. "
     )

### We now want to divide this existing training cohort into sub-populations

In [ ]:
save_path = "/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/"

### Create a training split containing only patients from London practices

In [ ]:
# # Create a split for London and the South East
# authority_group_1 = ["London"]

# group1_train_practice_ids = reduce_by_health_authority(
#     train_practice_ids, authority_group_1, collector
# )
# group1_val_practice_ids = reduce_by_health_authority(
#     val_practice_ids, authority_group_1, collector
# )
# group1_test_practice_ids = reduce_by_health_authority(
#     test_practice_ids, authority_group_1, collector
# )
# print(f"{', '.join(authority_group_1)} splits have {len(group1_train_practice_ids)}, "
#       f"{len(group1_val_practice_ids)} and {len(group1_test_practice_ids)} "
#       f"training, validation and test practices, respectively. "
#       )

# split_group1 = {"train": group1_train_practice_ids, "val": group1_val_practice_ids, "test": group1_test_practice_ids}
# with open(save_path + f'practice_id_splits_{"_".join(authority_group_1)}.pickle', 'wb') as handle:
#     pickle.dump(split_group1, handle, protocol=pickle.HIGHEST_PROTOCOL)


### Create a training split containing only patients from North Eastern practices

In [ ]:
# Create a split for the North East of England
authority_group_2 = ["North East"]

group2_train_practice_ids = reduce_by_health_authority(
    train_practice_ids, authority_group_2, collector
)
group2_val_practice_ids = reduce_by_health_authority(
    val_practice_ids, authority_group_2, collector
)
group2_test_practice_ids = reduce_by_health_authority(
    test_practice_ids, authority_group_2, collector
)
print(f"{', '.join(authority_group_2)} splits have {len(group2_train_practice_ids)}, "
       f"{len(group2_val_practice_ids)} and {len(group2_test_practice_ids)} "
       f"training, validation and test practices, respectively. "
     )

split_group2 = {"train": group2_train_practice_ids, "val": group2_val_practice_ids, "test": group2_test_practice_ids}
with open(save_path + f'practice_id_splits_{"_".join(authority_group_2)}.pickle', 'wb') as handle:
    pickle.dump(split_group2, handle, protocol=pickle.HIGHEST_PROTOCOL)
